In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================
# CONFIG
# =====================
EMBED_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 2

PAD, SOS, EOS, UNK = 0, 1, 2, 3


# =====================
# VOCAB
# =====================
class Vocab:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def build(self, sentences):
        for sent in sentences:
            for w in sent.split():
                if w not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[w] = idx
                    self.idx2word[idx] = w


vocab = Vocab()

# =====================
# DATA
# =====================
sentences = [
    "hello how are you",
    "i am fine thank you",
    "what is your name",
    "my name is bot"
]

vocab.build(sentences)
VOCAB_SIZE = len(vocab.word2idx)


# =====================
def encode(text):
    return [vocab.word2idx.get(w, UNK) for w in text.split()]


def pad(seq, max_len):
    return seq + [PAD] * (max_len - len(seq))


# =====================
# MODEL
# =====================
class NextWordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_SIZE, EMBED_SIZE, padding_idx=PAD)
        self.lstm = nn.LSTM(EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, batch_first=True)
        self.fc = nn.Linear(HIDDEN_SIZE, VOCAB_SIZE)

    def forward(self, x):
        x = self.emb(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # 🔥 берем последний hidden state
        return out


model = NextWordModel().to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = optim.Adam(model.parameters())


# =====================
# BUILD TRAIN DATA
# =====================
def build_data(sentences):
    data = []

    for sent in sentences:
        words = sent.split()

        for i in range(1, len(words)):
            inp = " ".join(words[:i])
            target = words[i]

            data.append((inp, target))

    return data


train_data = build_data(sentences)


# =====================
# TRAIN
# =====================
def train():
    print("\n🔥 Training...\n")

    for epoch in range(100):
        total_loss = 0

        for inp_text, target_word in train_data:

            inp = torch.tensor(
                [pad(encode(inp_text), max_len=10)],
                device=device
            )

            target = torch.tensor(
                [vocab.word2idx.get(target_word, UNK)],
                device=device
            )

            optimizer.zero_grad()

            out = model(inp)

            loss = criterion(out, target)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if epoch % 10 == 0:
            print(f"epoch {epoch} loss {total_loss:.4f}")


# =====================
# PREDICT NEXT WORD
# =====================
def predict(text):
    model.eval()

    with torch.no_grad():
        inp = torch.tensor(
            [pad(encode(text), max_len=10)],
            device=device
        )

        out = model(inp)

        idx = out.argmax(1).item()

        return vocab.idx2word.get(idx, "<UNK>")


# =====================
# CHAT STYLE PREDICT
# =====================
def chat():
    print("\n💬 Next-word predictor (type exit)\n")

    while True:
        text = input("You: ")

        if text == "exit":
            break

        print("Next word:", predict(text))


# =====================
if __name__ == "__main__":
    train()
    chat()


🔥 Training...

epoch 0 loss 38.0990
epoch 10 loss 14.8072
epoch 20 loss 5.5755
epoch 30 loss 0.4958
epoch 40 loss 0.1666
epoch 50 loss 0.0931
epoch 60 loss 0.0618
epoch 70 loss 0.0447
epoch 80 loss 0.0341
epoch 90 loss 0.0270

💬 Next-word predictor (type exit)

